This code reads a structured concentration file and calculates the median values for each point per year

Author: Pietro Mazzon
Mail: pietro.mazzon@polimi.it
Date: 06/11/2024

In [1]:
# import libraries
import os

import pandas as pd
print('Pandas version: '+pd.__version__) #check version

import numpy as np
print('Numpy version: '+np.__version__) #check version

Pandas version: 2.1.4
Numpy version: 1.26.3


In [ ]:
# set WD
# cwd = os.getcwd()
# print(cwd)
cwd = "c:/Users/user/OneDrive - Politecnico di Milano/SF2-Inquinamento_diffuso/Elaborazioni/E_AnalisiChimiche/" #my directory

# set input files dir
filepath = os.path.join(cwd,'PerConfronto')
print('La cartella dei file di input è: ',filepath)

La cartella dei file di input è:  c:/Users/user/OneDrive - Politecnico di Milano/SF2-Inquinamento_diffuso/Elaborazioni/E_AnalisiChimiche/PerConfronto


In [3]:
# Read the DF
file = 'idrochimica_tutti_step6_SL_31102024'
extension = '.xlsx'
path = os.path.join(filepath,file + extension)
df = pd.read_excel(path)

In [4]:
# Writes DF as csv
writepath = os.path.join(filepath,file + '.csv')
df.to_csv(writepath, index=False)

In [5]:
# explore DF

#df.head() # firts records of DF
#df.tail() # last records of DF
# df.shape #print dimensions of DF
df.dtypes # variable types
#df.describe()
#print(df.columns)

COMUNE                          object
ID_PUNTO                        object
DATA                    datetime64[ns]
PUNTO_PRELIEVO                  object
Descrizione Punto               object
Tipo di campione                object
Tipologia di analisi            object
Nota Prelievo                   object
Nota Prelevatore                object
VALORE_ORIGINE                  object
VALORE_MODIFICATO              float64
PARAMETRO                       object
UM                              object
FONTE                           object
dtype: object

In [5]:
# drop unwanted cols

#df2=df.copy() # for testing

df.drop(
    ['COMUNE', 'Descrizione Punto', 'PUNTO_PRELIEVO', 'Tipo di campione', 'Tipologia di analisi', 
     'Nota Prelievo', 'Nota Prelevatore', 'VALORE_ORIGINE', 'UM', 'FONTE'],
    axis=1, 
    inplace=True
)


print(df.columns)

Index(['ID_PUNTO', 'DATA', 'VALORE_MODIFICATO', 'PARAMETRO'], dtype='object')


In [7]:
# check for NaN values

# Checking for missing values using isnull()
df.isnull() # Creates a set of boolean values
df.isnull().sum() # make the sum to count the missing values -> better, summarize

ID_PUNTO             0
DATA                 0
VALORE_MODIFICATO    0
PARAMETRO            0
dtype: int64

In [ ]:
# groupby ID_PUNTO and then by YEAR
#df2=df.copy() # for testing
# create a column "YEAR"
df['ANNO']=df['DATA'].dt.year
#df.describe()
#df.head()
#df.tail()

# groupby and median values
result_grouped = df.groupby(['PARAMETRO','ID_PUNTO','ANNO'])['VALORE_MODIFICATO'].median() #select by PARAMETER, point and by year, then calculates the median of the column valore modificato

result_reset_index = result_grouped.reset_index() # make the series "result" a DF and reset of indexes
result_reset_index.rename(columns={'VALORE_MODIFICATO': 'MEDIANA_ANNO'}, inplace=True)
print(result_reset_index.head(10))

     PARAMETRO   ID_PUNTO  ANNO  MEDIANA_ANNO
0  Cloroformio      77087  2016          0.50
1  Cloroformio  150930097  2018          0.70
2  Cloroformio  150930097  2019          0.60
3  Cloroformio  150930097  2020          0.60
4  Cloroformio  150930097  2021          0.60
5  Cloroformio  150930097  2022          0.55
6  Cloroformio  150930098  2018          0.25
7  Cloroformio  150930098  2019          0.25
8  Cloroformio  150930098  2020          0.25
9  Cloroformio  150930098  2021          0.25


In [ ]:
#writepath2 = os.path.join(filepath,'mediane.csv')
result_reset_index.to_excel(os.path.join(filepath, "mediane_annuale_anagrafica_all.xlsx")) # save to excel, in the same filepath as the input file
#print(result_reset_index)
